<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/phase9_portfolio_launch_02_fria_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 9: FRIA Template
**Goal**: Build a general-purpose Fundamental Rights Impact
      Assessment generator that works for any audited AI
      system, not only the Phase 8 TalentMatch AI case, with
      an evidence-level distinction (automated vs manually
      attested) per required field.
**Regulatory mapping**: EU AI Act Article 27, UK SM&CR,
                    UK DUAA s.80.
**Date**: June 2026.
**Status**: In Progress.

In [1]:
import json
from datetime import datetime
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Any

# - EVIDENCE LEVELS -
AUTOMATED = "AUTOMATED"
ATTESTED = "ATTESTED"
MISSING = "MISSING"


@dataclass
class FRIARequirement:
  """One of the four required FRIA fields, with its evidence trail."""
  value: str
  evidence_level: str = ATTESTED
  source_reference: Optional[str] = None

  def is_satisfied(self) -> bool:
    return self.evidence_level != MISSING and bool(self.value)


@dataclass
class FRIAInput:
  """Generic FRIA input. Works for any audited system, any audit pipeline.
  One FRIAInput represents ONE accountability decision, i.e. one critical
  breach and the human response to it, not a whole audit. An audit with
  multiple breaches produces multiple FRIAInput objects, since Article 27
  treats each fundamental-rights-affecting decision as individually
  accountable, not bundled into a single record."""
  system_name: str
  system_purpose: str
  sector: str
  breach_reference: str
  accountable_individual: FRIARequirement
  deployment_timestamp: FRIARequirement
  pre_commitment_rationale: FRIARequirement
  interrogation_responses: List[FRIARequirement] = field(default_factory=list)


# - ADAPTER: REAL AUDIT LOG TO FRIA INPUT(S) -
def from_audit_log(system, audit_log_entries):
  """Returns a LIST of FRIAInput, one per critical breach in the log.
  Each breach is paired with the next HUMAN_AUTHORIZATION or HUMAN_ABORT
  event that follows it, since Phase 8's pipeline always logs the human
  decision immediately after the breach it responds to."""
  system_name = system.get("system_name") or system.get("name", "Unknown System")
  fria_inputs = []

  for i, entry in enumerate(audit_log_entries):
    if entry["event_type"] != "KILL_SWITCH_TRIGGERED":
      continue

    breach_entry = entry
    decision_entry = None
    for later_entry in audit_log_entries[i + 1:]:
      if later_entry["event_type"] in ("HUMAN_AUTHORIZATION", "HUMAN_ABORT"):
        decision_entry = later_entry
        break

    if decision_entry:
      accountable_individual = FRIARequirement(
          value=decision_entry.get("authorized_by") or "Unknown",
          evidence_level=AUTOMATED,
          source_reference=f"audit_log entry #{decision_entry['entry_number']}, "
                            f"hash {decision_entry['entry_hash'][:12]}..."
      )
      deployment_timestamp = FRIARequirement(
          value=decision_entry["timestamp"],
          evidence_level=AUTOMATED,
          source_reference=f"audit_log entry #{decision_entry['entry_number']}, "
                            f"previous_hash {decision_entry['previous_hash'][:12]}..."
      )
      rationale_text = decision_entry.get("details", "")
      pre_commitment_rationale = FRIARequirement(
          value=rationale_text if rationale_text else "No rationale text captured for this decision.",
          evidence_level=AUTOMATED if rationale_text else MISSING,
          source_reference=f"audit_log entry #{decision_entry['entry_number']}" if rationale_text else None,
      )
    else:
      accountable_individual = FRIARequirement(value="", evidence_level=MISSING)
      deployment_timestamp = FRIARequirement(value="", evidence_level=MISSING)
      pre_commitment_rationale = FRIARequirement(value="", evidence_level=MISSING)

    interrogation_response = FRIARequirement(
        value=breach_entry.get("details", ""),
        evidence_level=AUTOMATED,
        source_reference=f"audit_log entry #{breach_entry['entry_number']}, "
                          f"hash {breach_entry['entry_hash'][:12]}..."
    )

    fria_inputs.append(FRIAInput(
        system_name=system_name,
        system_purpose=system.get("purpose", "Not specified"),
        sector=system.get("sector", "Not specified"),
        breach_reference=f"{breach_entry['agent']} (entry #{breach_entry['entry_number']})",
        accountable_individual=accountable_individual,
        deployment_timestamp=deployment_timestamp,
        pre_commitment_rationale=pre_commitment_rationale,
        interrogation_responses=[interrogation_response],
    ))

  return fria_inputs


# - MANUAL INTAKE: REAL CLIENT CONVERSATION TO FRIA INPUT -
# Turns a real client conversation into a well-formed FRIAInput, instead
# of requiring you to hand-write a FRIARequirement for each field. Built
# for the code-only intake stage: Afrispan's Stage 1 personal delivery
# tool, before a real intake form exists (Stage 2 of the build plan).
#
# The arguments map to four intake questions, asked in this order:
#   1. Who is the named accountable person?
#   2. When was the decision made / system deployed?
#   3. What is their position, asked BEFORE the full breach is disclosed?
#   4. What is their response, asked AFTER the specific breach is
#      disclosed to them?
# Order matters: question 3 must be recorded before question 4 discloses
# the finding, or the rationale stops being pre-commitment and becomes a
# justification written with the outcome already known.
#
# Every *_source argument is required, not optional. An answer with no
# stated source is marked MISSING, even if a value was given, since an
# unsourced claim is not evidence.

def intake_fria(
    system_name: str,
    system_purpose: str,
    sector: str,
    breach_reference: str,
    accountable_name: str,
    accountable_source: str,
    decision_timestamp: str,
    timestamp_source: str,
    pre_commitment_rationale: str,
    rationale_source: str,
    interrogation_response: str,
    interrogation_source: str,
) -> FRIAInput:
  """Build a FRIAInput from manually-collected intake answers."""

  def build_field(value, source):
    value = (value or "").strip()
    source = (source or "").strip()
    if not value or not source:
      return FRIARequirement(value=value, evidence_level=MISSING)
    return FRIARequirement(value=value, evidence_level=ATTESTED, source_reference=source)

  return FRIAInput(
      system_name=system_name,
      system_purpose=system_purpose,
      sector=sector,
      breach_reference=breach_reference,
      accountable_individual=build_field(accountable_name, accountable_source),
      deployment_timestamp=build_field(decision_timestamp, timestamp_source),
      pre_commitment_rationale=build_field(pre_commitment_rationale, rationale_source),
      interrogation_responses=[build_field(interrogation_response, interrogation_source)],
  )


# - THE FRIA GENERATOR -
def build_fria(fria_input):
  requirements = {
      "accountable_individual": fria_input.accountable_individual,
      "deployment_timestamp": fria_input.deployment_timestamp,
      "pre_commitment_rationale": fria_input.pre_commitment_rationale,
  }

  all_satisfied = all(r.is_satisfied() for r in requirements.values())
  all_satisfied = all_satisfied and all(
      r.is_satisfied() for r in fria_input.interrogation_responses
  )

  evidence_summary = {name: req.evidence_level for name, req in requirements.items()}
  evidence_summary["interrogation_responses"] = [
      r.evidence_level for r in fria_input.interrogation_responses
  ]

  automated_count = sum(1 for r in requirements.values() if r.evidence_level == AUTOMATED)
  automated_count += sum(1 for r in fria_input.interrogation_responses if r.evidence_level == AUTOMATED)
  total_fields = len(requirements) + len(fria_input.interrogation_responses)

  return {
      "system_name": fria_input.system_name,
      "system_purpose": fria_input.system_purpose,
      "sector": fria_input.sector,
      "breach_reference": fria_input.breach_reference,
      "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
      "requirements": {name: asdict(req) for name, req in requirements.items()},
      "interrogation_responses": [asdict(r) for r in fria_input.interrogation_responses],
      "all_requirements_satisfied": all_satisfied,
      "evidence_summary": evidence_summary,
      "automated_evidence_ratio": f"{automated_count}/{total_fields}",
      "maturity_signal": (
          "Level 4+ indicators present (automated evidence trail)"
          if automated_count == total_fields
          else "Level 3 or below (manual attestation, no automated trail)"
          if automated_count == 0
          else "Mixed evidence (partial automated trail)"
      ),
  }


def render_fria_markdown(fria_record):
  def render_requirement(label, req_dict):
    level = req_dict["evidence_level"]
    badge = {"AUTOMATED": "[AUTOMATED EVIDENCE]",
              "ATTESTED": "[MANUAL ATTESTATION]",
              "MISSING": "[NOT SATISFIED]"}[level]
    lines = [f"### {label} {badge}", "", req_dict["value"] or "*Not provided.*"]
    if req_dict.get("source_reference"):
      lines.append(f"\n*Source: {req_dict['source_reference']}*")
    return "\n".join(lines)

  sections = [
      f"# Fundamental Rights Impact Assessment",
      f"**System:** {fria_record['system_name']}",
      f"**Purpose:** {fria_record['system_purpose']}",
      f"**Sector:** {fria_record['sector']}",
      f"**Breach addressed:** {fria_record['breach_reference']}",
      f"**Generated:** {fria_record['generated_at']}",
      f"**Evidence ratio:** {fria_record['automated_evidence_ratio']} fields automated",
      f"**Maturity signal:** {fria_record['maturity_signal']}",
      "", "---", "",
      render_requirement("1. Named Accountable Individual",
                          fria_record["requirements"]["accountable_individual"]),
      "",
      render_requirement("2. Tamper-Evident Deployment Timestamp",
                          fria_record["requirements"]["deployment_timestamp"]),
      "",
      render_requirement("3. Pre-Commitment Rationale",
                          fria_record["requirements"]["pre_commitment_rationale"]),
      "",
      "### 4. Structured Interrogation Protocol", "",
  ]

  for i, resp in enumerate(fria_record["interrogation_responses"], 1):
    sections.append(render_requirement(f"Response {i}", resp))
    sections.append("")

  sections.append("---")
  status = "ALL REQUIREMENTS SATISFIED" if fria_record["all_requirements_satisfied"] \
           else "INCOMPLETE, ONE OR MORE REQUIREMENTS NOT SATISFIED"
  sections.append(f"\n**Overall status: {status}**")

  return "\n".join(sections)


# - DEMONSTRATION 1: FROM THE REAL PHASE 8 LESSON 3 AUDIT LOG -
# These are the genuine 12 entries from your actual Lesson 3 run, with
# full SHA-256 hashes, not placeholders. Two critical breaches occurred
# (bias, then oversight), each followed by its own human authorization.
# from_audit_log() correctly returns one FRIA per breach, not one for
# the whole audit.
print("====== DEMONSTRATION 1: FRIA(s) FROM REAL PHASE 8 LESSON 3 LOG ======\n")

real_audit_log = [
    {"entry_number": 1, "timestamp": "2026-06-16 14:43:20", "event_type": "AUDIT_STARTED",
     "agent": "Governance Controller", "details": "Auditing TalentMatch AI.Sector: employment.",
     "authorized_by": "Governance Controller", "previous_hash": "GENESIS",
     "entry_hash": "b3d6683215a6b11cab0a3409b2cfe14535c8bd230df47d01547577193769cd4e"},
    {"entry_number": 2, "timestamp": "2026-06-16 14:43:20", "event_type": "AGENT_STARTED",
     "agent": "bias", "details": "Running bias audit agent.", "authorized_by": None,
     "previous_hash": "b3d6683215a6b11cab0a3409b2cfe14535c8bd230df47d01547577193769cd4e",
     "entry_hash": "073dc80ad6b6b49d4dde4f5c44509ec5893c9e99307b41c406f8a1c759206f33"},
    {"entry_number": 3, "timestamp": "2026-06-16 14:43:20", "event_type": "AGENT_COMPLETED",
     "agent": "Bias Audit Agent", "details": "Verdict: FAIL.Article: EU AI Act Article 10.",
     "authorized_by": None,
     "previous_hash": "073dc80ad6b6b49d4dde4f5c44509ec5893c9e99307b41c406f8a1c759206f33",
     "entry_hash": "8bae181de688928f6e58542af28f81bf80d3306b4c0256ad072e9e116e51fc23"},
    {"entry_number": 4, "timestamp": "2026-06-16 14:43:20", "event_type": "KILL_SWITCH_TRIGGERED",
     "agent": "Bias Audit Agent", "details": "Critical breach detected. EU AI Act Article 10.System halted immediately.",
     "authorized_by": None,
     "previous_hash": "8bae181de688928f6e58542af28f81bf80d3306b4c0256ad072e9e116e51fc23",
     "entry_hash": "10d1c68516911366fa9ddde0c0c3f0ed18ce5c1ccd6145d19deeb6c983429d3b"},
    {"entry_number": 5, "timestamp": "2026-06-16 14:43:20", "event_type": "HUMAN_AUTHORIZATION",
     "agent": "Human Auditor",
     "details": "position: Critical breach confirmed.Audit to continue under supervision pending full remediation plan.",
     "authorized_by": "Steve Onyeke, Lead AI Auditor, Afrispan Data Labs",
     "previous_hash": "10d1c68516911366fa9ddde0c0c3f0ed18ce5c1ccd6145d19deeb6c983429d3b",
     "entry_hash": "4eba89c9b220bc670510b6d4371fbcff9d3c4c1aae9e6be4e41ef6d37e88c143"},
    {"entry_number": 6, "timestamp": "2026-06-16 14:43:20", "event_type": "AGENT_STARTED",
     "agent": "oversight", "details": "Running oversight audit agent.", "authorized_by": None,
     "previous_hash": "4eba89c9b220bc670510b6d4371fbcff9d3c4c1aae9e6be4e41ef6d37e88c143",
     "entry_hash": "2470f2d18845308a04b391255cbcb49bc8336f0d8d380f5a44578ddf3fb1245c"},
    {"entry_number": 7, "timestamp": "2026-06-16 14:43:20", "event_type": "AGENT_COMPLETED",
     "agent": "Oversight Audit Agent", "details": "Verdict: FAIL.Article: EU AI Act Article 14.",
     "authorized_by": None,
     "previous_hash": "2470f2d18845308a04b391255cbcb49bc8336f0d8d380f5a44578ddf3fb1245c",
     "entry_hash": "5e6a5f7b40598c304f52e20cf37c2abe4f90ec10cf0a6cb95476895dcbd26f00"},
    {"entry_number": 8, "timestamp": "2026-06-16 14:43:20", "event_type": "KILL_SWITCH_TRIGGERED",
     "agent": "Oversight Audit Agent", "details": "Critical breach detected. EU AI Act Article 14.System halted immediately.",
     "authorized_by": None,
     "previous_hash": "5e6a5f7b40598c304f52e20cf37c2abe4f90ec10cf0a6cb95476895dcbd26f00",
     "entry_hash": "18e78f8f01251a45034c79c30b7b39134addb0f0e775168e6cbe5405011b7613"},
    {"entry_number": 9, "timestamp": "2026-06-16 14:43:20", "event_type": "HUMAN_AUTHORIZATION",
     "agent": "Human Auditor",
     "details": "position: Critical breach confirmed.Audit to continue under supervision pending full remediation plan.",
     "authorized_by": "Steve Onyeke, Lead AI Auditor, Afrispan Data Labs",
     "previous_hash": "18e78f8f01251a45034c79c30b7b39134addb0f0e775168e6cbe5405011b7613",
     "entry_hash": "079c2dff6e1842792ac83e171691847496ac9c622c42bc78ab8b69668cc4150e"},
    {"entry_number": 10, "timestamp": "2026-06-16 14:43:20", "event_type": "AGENT_STARTED",
     "agent": "documentation", "details": "Running documentation audit agent.", "authorized_by": None,
     "previous_hash": "079c2dff6e1842792ac83e171691847496ac9c622c42bc78ab8b69668cc4150e",
     "entry_hash": "8a7d7e0ae115de87c2846f43eea7e695346c0fd6275671bba6bbf45bdc3e78b1"},
    {"entry_number": 11, "timestamp": "2026-06-16 14:43:20", "event_type": "AGENT_COMPLETED",
     "agent": "Documentation Audit Agent", "details": "Verdict: FAIL.Article: EU AI Act Article 11 and 18.",
     "authorized_by": None,
     "previous_hash": "8a7d7e0ae115de87c2846f43eea7e695346c0fd6275671bba6bbf45bdc3e78b1",
     "entry_hash": "326a167657733472a825b76464240671ece18073c8bd2bac405d30efd8918d97"},
    {"entry_number": 12, "timestamp": "2026-06-16 14:43:20", "event_type": "AUDIT_COMPLETED",
     "agent": "Governance Controller",
     "details": "Audit completed. Agents run: 3. Agents halted: 0. Overall: CRITICAL NON-COMPLIANCE.",
     "authorized_by": None,
     "previous_hash": "326a167657733472a825b76464240671ece18073c8bd2bac405d30efd8918d97",
     "entry_hash": "7de55fa4f7f391fc52a874bdaee051ba76ad7171f0503661b78c20c764090a95"},
]

real_system = {
    "system_name": "TalentMatch AI",
    "purpose": "Automated CV screening",
    "sector": "employment",
}

# To load this from your saved Drive file instead of the inline data
# above, replace the real_audit_log assignment with:
#   with open(SAVE_PATH + "immutable_audit_log.json", "r") as f:
#       real_audit_log = json.load(f)

fria_inputs = from_audit_log(real_system, real_audit_log)
print(f"Number of FRIA documents generated: {len(fria_inputs)}\n")

records = []
for fria_input in fria_inputs:
  record = build_fria(fria_input)
  records.append(record)
  print(render_fria_markdown(record))
  print("\n" + "=" * 60 + "\n")


# - DEMONSTRATION 2: A REAL CLIENT INTAKE CONVERSATION -
# This is the realistic shape of a code-only Afrispan engagement today:
# a named accountable person at the client answers four questions, and
# intake_fria() turns those real answers into a complete FRIA, with no
# automated audit log behind it at all.
print("\n====== DEMONSTRATION 2: CLIENT INTAKE, XYZ LOANAPPROVE AI ======\n")

xyz_fria = intake_fria(
    system_name="XYZ LoanApprove AI",
    system_purpose="Automated approval decisions for personal loans",
    sector="financial services",
    breach_reference="Bias Audit: gender disparate impact 0.38, fails Article 10",

    accountable_name="Adjoa Mensah, Head of Risk and Compliance, XYZ Ltd",
    accountable_source="Intake interview, 2026-06-25, recorded with consent",

    decision_timestamp="2026-06-25 10:30:00",
    timestamp_source="XYZ deployment ticket DEP-2291",

    pre_commitment_rationale=(
        "Loan approval models commonly show some disparity given income "
        "correlation with protected characteristics. We believe this is "
        "explainable by legitimate underwriting factors, but want a "
        "remediation plan before full rollout."
    ),
    rationale_source="Verbal statement, captured before bias audit results shared",

    interrogation_response=(
        "Gender disparate impact of 0.38 falls below the 0.80 threshold "
        "under Article 10. We commit to a remediation review within 30 "
        "days and will not proceed to full production rollout until "
        "ratio reaches at least 0.80."
    ),
    interrogation_source="Follow-up statement after breach details disclosed, 2026-06-25",
)

xyz_record = build_fria(xyz_fria)
records.append(xyz_record)
print(render_fria_markdown(xyz_record))


# - DEMONSTRATION 3: HONESTY CHECK, A FIELD WITH NO SOURCE -
# Confirms that an answer given without a stated source is correctly
# flagged MISSING, not silently accepted as ATTESTED. A real engagement
# should never produce a record this generator would call satisfied
# unless every field has both a value AND a traceable source.
print("\n\n====== DEMONSTRATION 3: HONESTY CHECK, MISSING SOURCE ======\n")

incomplete_fria = intake_fria(
    system_name="Untested System",
    system_purpose="Example for validation testing",
    sector="example",
    breach_reference="Example breach",
    accountable_name="Someone Somewhere",
    accountable_source="",  # no source given
    decision_timestamp="2026-06-25",
    timestamp_source="Verbal mention, not documented",
    pre_commitment_rationale="",  # no value given
    rationale_source="",
    interrogation_response="A response without any stated source.",
    interrogation_source="",  # no source given
)

incomplete_record = build_fria(incomplete_fria)
print(render_fria_markdown(incomplete_record))
print("\nExpected: accountable_individual MISSING (no source), "
      "pre_commitment_rationale MISSING (no value), "
      "interrogation response MISSING (no source despite having a value).")


print("\n\n====== COMPARISON ======")
for i, r in enumerate(records, 1):
  print(f"Record {i} ({r['breach_reference']}): "
        f"{r['automated_evidence_ratio']} automated, {r['maturity_signal']}")
print(f"Honesty check ({incomplete_record['breach_reference']}): "
      f"{incomplete_record['automated_evidence_ratio']} automated, "
      f"{incomplete_record['maturity_signal']}")

====== DEMONSTRATION 1: FRIA(s) FROM REAL PHASE 8 LESSON 3 LOG ======

Number of FRIA documents generated: 2

# Fundamental Rights Impact Assessment
**System:** TalentMatch AI
**Purpose:** Automated CV screening
**Sector:** employment
**Breach addressed:** Bias Audit Agent (entry #4)
**Generated:** 2026-06-25 09:22:44
**Evidence ratio:** 4/4 fields automated
**Maturity signal:** Level 4+ indicators present (automated evidence trail)

---

### 1. Named Accountable Individual [AUTOMATED EVIDENCE]

Steve Onyeke, Lead AI Auditor, Afrispan Data Labs

*Source: audit_log entry #5, hash 4eba89c9b220...*

### 2. Tamper-Evident Deployment Timestamp [AUTOMATED EVIDENCE]

2026-06-16 14:43:20

*Source: audit_log entry #5, previous_hash 10d1c6851691...*

### 3. Pre-Commitment Rationale [AUTOMATED EVIDENCE]

position: Critical breach confirmed.Audit to continue under supervision pending full remediation plan.

*Source: audit_log entry #5*

### 4. Structured Interrogation Protocol

### Response 1 [AUT

## Findings: FRIA Template

**Generator type:** General-purpose, accepts automated audit
                     evidence or manual client intake
**FRIA documents generated:** 4 (2 from real Phase 8 audit log,
                     1 from realistic client intake, 1 honesty check)
**Date:** June 2026
**Regulatory mapping:** EU AI Act Article 27, UK SM&CR, DUAA s.80

### What Was Built

A FRIA generator with two ways to populate the same underlying
structure. from_audit_log() auto-populates from a real Phase 8
immutable audit log with AUTOMATED evidence and zero manual
re-entry. intake_fria() turns a real client conversation, four
questions asked in a specific order, into a well-formed FRIA
with ATTESTED evidence, for engagements with no automated audit
trail at all.

One FRIAInput represents one accountability decision, a single
critical breach and the human response to it, not a whole audit.
Both demonstrations confirm this: the real Phase 8 log produced
two independent FRIA documents, one per breach, not one bundled
record for the whole audit.

### Results

| Record | Breach | Evidence | Status |
|---|---|---|---|
| Phase 8, bias | Article 10 (entry #4/#5) | 4/4 AUTOMATED | SATISFIED |
| Phase 8, oversight | Article 14 (entry #8/#9) | 4/4 AUTOMATED | SATISFIED |
| XYZ LoanApprove AI | Article 10, gender DI 0.38 | 0/4 AUTOMATED, 4/4 ATTESTED | SATISFIED |
| Honesty check | Example, deliberately incomplete | 0/4 | INCOMPLETE |

### Key Findings

1. intake_fria() requires both a value and a source for every
   field. The honesty check confirms this is enforced correctly:
   "Someone Somewhere" was typed in as a name, but with no
   source given, the field is marked NOT SATISFIED, not
   ATTESTED. A value alone is not evidence.

2. The two intake paths, automated and manual, produce
   structurally identical output. A client reading a Phase 8
   FRIA and an XYZ-style manually-attested FRIA sees the same
   document format, with the evidence_level badges making the
   difference in rigor visible rather than hidden.

3. This closes the loop on the real-world question of how
   Afrispan actually delivers a FRIA to a client: the four
   intake questions (named accountable individual, decision
   timestamp, pre-commitment rationale asked before disclosure,
   structured response after disclosure) map directly onto
   intake_fria()'s four required value/source pairs. No
   translation step needed between a real conversation and a
   working FRIA document.

### Recommendation
Proceed to the Conformity Report generator, designed the same
way: general-purpose input, evidence-level distinction per
field, an automated adapter, and a manual intake function for
engagements without an automated audit trail.